In [129]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [130]:
from src.shared.json_tools import load_json_long
from paths import DATA_DIR
import os

known_attack_dataset = []
directory = DATA_DIR / 'commands'

def recursive_load_jsons(directory):
    for entry in os.scandir(directory):
        if entry.is_dir():
            recursive_load_jsons(entry.path)
        elif entry.is_file() and entry.name.endswith('.json'):
            try:
                data = load_json_long(entry.path)
                known_attack_dataset.append(data)
            except Exception as e:
                print(f"Failed to load {entry.path}: {e}")

recursive_load_jsons(directory)

In [131]:
import random as rd
def convert_command_entry(entry):
    try:
        timestamp = ''

        # Extract command and arguments
        cmd_parts = entry['cmd'].split()
        pid = rd.randint(1000, 99999)

        # Base structure
        syscall_entry = {
            "timestamp": timestamp,
            "success": 1,
            "uid": "1000" if entry.get("username") != "root" else "0",
            "euid": "0" if entry.get("username") == "root" else "1000",
            "syscall": "0",
            "ppid": pid,
            "pid": pid,
            "command": cmd_parts[0],
            "arguments": cmd_parts,
            "CWD": entry.get("wd", "/")
        }

        return syscall_entry
    except Exception as e:
        return 0

dt = [[convert_command_entry(entry) for entry in logs] for logs in known_attack_dataset]

In [132]:
dt[0][0]

{'timestamp': '',
 'success': 1,
 'uid': '0',
 'euid': '0',
 'syscall': '0',
 'ppid': 64522,
 'pid': 64522,
 'command': 'nmap',
 'arguments': ['nmap', '172.18.1.5'],
 'CWD': '/root'}

In [133]:
dt = [[{"target": 1, 'content': entry} for entry in logs] for logs in dt]

In [134]:
from src.shared.json_tools import write_json_long
from paths import ROOT_DIR

for i, logs in enumerate(dt):
    write_json_long(logs, ROOT_DIR / 'new_data_raw/commands' / f'{i}.json')

In [135]:
command_data = dt.copy()
del dt

In [136]:
apt_data = [load_json_long(i) for i in (ROOT_DIR / 'new_data_raw/apt').glob('*.json')]

In [137]:
len(apt_data)

13

In [138]:
len(command_data)

267

In [139]:
sum([len(i) for i in command_data])

21108

In [140]:
sum([len(i) for i in apt_data])

83484

In [141]:
## check how much positive classes we have
sum([len([i for i in logs if i['target']>=0.5]) for logs in apt_data])

11632

In [142]:
# so we have 71852 neg and 32740 pos

In [143]:
import numpy as np
np.mean([len(i) for i in command_data])

np.float64(79.0561797752809)

In [144]:
np.mean([len(i) for i in apt_data])

np.float64(6421.846153846154)

In [145]:
for i, logs in enumerate(apt_data):
    print(f'{i} {len(logs)}')

0 8478
1 9029
2 6510
3 49
4 1
5 2
6 7297
7 9367
8 8556
9 9667
10 7220
11 7713
12 9595


In [146]:
small_apt = [logs for logs in apt_data if len(logs) < 1000]

In [147]:
len(small_apt)

3

In [148]:
a = []
for i in small_apt: a += i

In [149]:
small_apt = a.copy()

In [150]:
apt_data = [logs for logs in apt_data if len(logs) >= 1000]

In [151]:
for i, logs in enumerate(apt_data):
    print(f'{i} {len(logs)}')

0 8478
1 9029
2 6510
3 7297
4 9367
5 8556
6 9667
7 7220
8 7713
9 9595


In [152]:
command_data += [small_apt]

In [153]:
len(command_data)

268

In [154]:
combos = [(logs, command_data[i*27: (i+1)*27]) for i, logs in enumerate(apt_data)]

In [155]:
for i, logs in enumerate(apt_data):
    print(f'{i} {len(logs)} - {len([i for i in logs if i["target"] >= 0.5])} positive - {len([i for i in logs if i["target"] >= 0.5]) + 270} w extra')

0 8478 - 946 positive - 1216 w extra
1 9029 - 1550 positive - 1820 w extra
2 6510 - 855 positive - 1125 w extra
3 7297 - 1042 positive - 1312 w extra
4 9367 - 2165 positive - 2435 w extra
5 8556 - 886 positive - 1156 w extra
6 9667 - 1879 positive - 2149 w extra
7 7220 - 846 positive - 1116 w extra
8 7713 - 579 positive - 849 w extra
9 9595 - 884 positive - 1154 w extra


In [158]:
import random
from datetime import datetime, timedelta
import copy

def inject_malicious_commands(combo_tuple_outer):
    combo_tuple = copy.copy(combo_tuple_outer)
    logs, command_data = combo_tuple
    log_len = len(logs)

    # Convert all timestamps from UNIX to datetime
    for log in logs:
        ts = log['content']['timestamp']
        if isinstance(ts, str):
            ts = float(ts)
        log['content']['timestamp'] = datetime.fromtimestamp(ts)

    # Gather UID stats
    unique_uids = list(set(log['content']['uid'] for log in logs))
    uid_log_map = {uid: [log for log in logs if log['content']['uid'] == uid] for uid in unique_uids}
    total_logs = len(logs)

    injected_logs = []

    for uid in unique_uids:
        uid_logs = uid_log_map[uid]
        uid_log_count = len(uid_logs)
        proportion = uid_log_count / total_logs

        # How many command groups to inject
        num_injections = max(1, int(len(command_data) * proportion))

        if not uid_logs:
            continue

        for _ in range(num_injections):
            cmd_group = copy.deepcopy(random.choice(command_data))  # A list of logs
            insert_point = random.choice(uid_logs)
            base_time = insert_point['content']['timestamp']
            injection_start = base_time + timedelta(seconds=random.randint(0, 300))  # Start within 5 minutes

            for i, cmd in enumerate(cmd_group):
                try:
                    injected = {
                        "target": 1,
                        "id": 0,  # Placeholder ID
                        'content': {
                            **cmd['content'],
                            'uid': uid,
                            'timestamp': injection_start + timedelta(seconds=i)  # Space them by 1 sec
                        }
                    }
                    injected_logs.append(injected)
                except Exception as e:
                    print(cmd)

    # Combine and sort all logs by timestamp
    combined_logs = logs + injected_logs
    combined_logs.sort(key=lambda x: x['content']['timestamp'])

    # Ensure timestamps are strictly increasing
    shifted_logs = []
    last_time = None
    for log in combined_logs:
        if last_time and log['content']['timestamp'] <= last_time:
            log['content']['timestamp'] = last_time + timedelta(seconds=1)
        last_time = log['content']['timestamp']
        shifted_logs.append(log)

    # Convert timestamps back to UNIX string
    for log in shifted_logs:
        log['content']['timestamp'] = str(log['content']['timestamp'].timestamp())

    return shifted_logs


In [166]:
logs_w_injected = [sorted(inject_malicious_commands(i), key=lambda x: x['content']['timestamp']) for i in combos]

{'target': 1, 'content': 0}
{'target': 1, 'content': 0}
{'target': 1, 'content': 0}
{'target': 1, 'content': 0}
{'target': 1, 'content': 0}
{'target': 1, 'content': 0}
{'target': 1, 'content': 0}
{'target': 1, 'content': 0}
{'target': 1, 'content': 0}
{'target': 1, 'content': 0}
{'target': 1, 'content': 0}
{'target': 1, 'content': 0}
{'target': 1, 'content': 0}
{'target': 1, 'content': 0}
{'target': 1, 'content': 0}
{'target': 1, 'content': 0}
{'target': 1, 'content': 0}
{'target': 1, 'content': 0}
{'target': 1, 'content': 0}
{'target': 1, 'content': 0}
{'target': 1, 'content': 0}
{'target': 1, 'content': 0}
{'target': 1, 'content': 0}
{'target': 1, 'content': 0}
{'target': 1, 'content': 0}
{'target': 1, 'content': 0}
{'target': 1, 'content': 0}
{'target': 1, 'content': 0}
{'target': 1, 'content': 0}
{'target': 1, 'content': 0}


In [168]:
logs_w_injected = [[{"target": log['target'], "id": i, "content": log['content']} for i, log in enumerate(logs)] for logs in logs_w_injected]

In [169]:
len(logs_w_injected)

10

In [170]:
for i, logs in enumerate(logs_w_injected):
    print(f'{i} {len(logs)} - {len([i for i in logs if i["target"] >= 0.5])} positive')

0 10001 - 2469 positive
1 11076 - 3597 positive
2 8734 - 3079 positive
3 9434 - 3179 positive
4 13032 - 5830 positive
5 10739 - 3069 positive
6 11075 - 3287 positive
7 10046 - 3672 positive
8 10587 - 3453 positive
9 11050 - 2339 positive


In [171]:
for dt in logs_w_injected:
    write_json_long(dt, ROOT_DIR / 'unified_data' / f'{rd.randint(0, 100000)}.json')